# EDA

#### 1. Which months, days, and hours generate the most revenue, and how does average order value change over time?
#### 2. Which countries contribute the most to total revenue, and how does revenue distribution vary across geographic markets?


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

# Keep the source file, notebook, and database together in this project folder.
PROJECT_DIR = Path(r'E:\alchemist\task1')
excel_path = PROJECT_DIR / 'Online Retail.xlsx'
database_path = PROJECT_DIR / 'online_retail_db.sqlite'

if not excel_path.exists():
    raise FileNotFoundError(f'Source workbook not found: {excel_path}')

In [7]:
# openpyxl not working for some reason, used calamine instead
retail_df = pd.read_excel(excel_path, engine='calamine')  #Read excel file into dataframe

print(f'Loaded {retail_df.shape[0]:,} rows and {retail_df.shape[1]} columns')
display(retail_df.head())

Loaded 541,909 rows and 8 columns


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [8]:
# Store an unchanged copy of the loaded table in SQLite for portability.
with sqlite3.connect(database_path) as connection:
    retail_df.to_sql('online_retail', connection, if_exists='replace', index=False)

print(f'Saved {len(retail_df):,} rows to {database_path.name}, table online_retail.')

Saved 541,909 rows to online_retail_db.sqlite, table online_retail.


In [12]:
# Report the basic shape, column names, and current pandas data types
overview = pd.DataFrame({
    'data_type': retail_df.dtypes.astype(str),
    'non_missing_count': retail_df.notna().sum(),
    'unique_non_missing_values': retail_df.nunique(dropna=True),
    'missing_count': retail_df.isna().sum(),
    'missing_percent': (retail_df.isna().mean() * 100).round(2),
})
overview.index.name = 'column name'
print(f'Rows: {retail_df.shape[0]:,} | Columns: {retail_df.shape[1]}')
display(overview)

# Summarize each column without printing every value in high-cardinality columns.
display(retail_df.describe(include='all').T)

Rows: 541,909 | Columns: 8


,data_type,non_missing_count,unique_non_missing_values,missing_count,missing_percent
column name,,,,,
InvoiceNo,object,541909,25900,0,0.00
StockCode,object,541909,4070,0,0.00
Description,object,540455,4223,1454,0.27
Quantity,int64,541909,722,0,0.00
InvoiceDate,datetime64[us],541909,23260,0,0.00
UnitPrice,float64,541909,1630,0,0.00
CustomerID,float64,406829,4372,135080,24.93
Country,str,541909,38,0,0.00


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909.0,25900.0,573585.0,1114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
# Show all distinct values
unique_value_summary = {}
for column in retail_df.columns:
    distinct_values = pd.unique(retail_df[column].dropna())
    # print(len(distinct_values))
    # if len(distinct_values) <= 10: # avoid invoiceid and quantity
    #     unique_value_summary[column] = list(distinct_values)

    unique_value_summary[column] = {
        'number of unique': len(distinct_values),
        'first_10_examples': list(distinct_values[:10]),
    }

for column, values in unique_value_summary.items():
    print(f'\n{column}: {values}')

print("\n ========Any duplicate rows after composite primary key of InvoiceNo and StockCode:======== \n")
display(retail_df.duplicated(subset=['InvoiceNo', 'StockCode'], keep=False)) # Check for any duplicate rows after composite key


InvoiceNo: {'number of unique': 25900, 'first_10_examples': [536365, 536366, 536367, 536368, 536369, 536370, 536371, 536372, 536373, 536374]}

StockCode: {'number of unique': 4070, 'first_10_examples': ['85123A', 71053, '84406B', '84029G', '84029E', 22752, 21730, 22633, 22632, 84879]}

Description: {'number of unique': 4223, 'first_10_examples': ['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN', 'CREAM CUPID HEARTS COAT HANGER', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'RED WOOLLY HOTTIE WHITE HEART.', 'SET 7 BABUSHKA NESTING BOXES', 'GLASS STAR FROSTED T-LIGHT HOLDER', 'HAND WARMER UNION JACK', 'HAND WARMER RED POLKA DOT', 'ASSORTED COLOUR BIRD ORNAMENT']}

Quantity: {'number of unique': 722, 'first_10_examples': [np.int64(6), np.int64(8), np.int64(2), np.int64(32), np.int64(3), np.int64(4), np.int64(24), np.int64(12), np.int64(48), np.int64(18)]}

InvoiceDate: {'number of unique': 23260, 'first_10_examples': [np.datetime64('2010-12-01T08:26:00.000000'), np.datetime64('2010-

0         False
1         False
2         False
3         False
4         False
          ...  
541904    False
541905    False
541906    False
541907    False
541908    False
Length: 541909, dtype: bool

In [29]:
# Missing value columns
missing_summary = overview.loc[overview['missing_count'] > 0, ['missing_count', 'missing_percent']]
display(missing_summary.sort_values('missing_percent'))

print(f'Exact duplicate rows: {retail_df.duplicated().sum():,}') #Duplicate rows

# Show the observed date range and numeric ranges when those columns are available.
#if 'InvoiceDate' in retail_df.columns:
#    parsed_dates = pd.to_datetime(retail_df['InvoiceDate'], errors='coerce')
#    print(f'InvoiceDate parse failures: {parsed_dates.isna().sum():,}')
#    print(f'InvoiceDate range: {parsed_dates.min()} to {parsed_dates.max()}')

for column in ['Quantity', 'UnitPrice']:
    if column in retail_df.columns:
        print(f'\n{column} summary:')
        display(retail_df[column].describe())
        print(f'Negative values: {(retail_df[column] < 0).sum():,}; zero values: {(retail_df[column] == 0).sum():,}')

print("\n ========================================================================== \n")
display(retail_df.loc[retail_df['Quantity'] < 0, ["InvoiceNo", "Quantity"]].head(10))

,missing_count,missing_percent
column name,,
Description,1454,0.27
CustomerID,135080,24.93


Exact duplicate rows: 5,268

Quantity summary:


count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

Negative values: 10,624; zero values: 0

UnitPrice summary:


count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

Negative values: 2; zero values: 2,515




,InvoiceNo,Quantity
141,C536379,-1
154,C536383,-1
235,C536391,-12
236,C536391,-24
237,C536391,-24
238,C536391,-24
239,C536391,-12
240,C536391,-12
241,C536391,-24
939,C536506,-6
